In [1]:
import pandas as pd
p = "../data/" 
orders = pd.read_csv(p+"olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp","order_delivered_customer_date","order_estimated_delivery_date"])
items = pd.read_csv(p+"olist_order_items_dataset.csv")
products = pd.read_csv(p+"olist_products_dataset.csv")
cats = pd.read_csv(p+"product_category_name_translation.csv")
customers = pd.read_csv(p+"olist_customers_dataset.csv")
reviews = pd.read_csv(p+"olist_order_reviews_dataset.csv")

orders = orders[orders.order_status == "delivered"]
orders = orders[(orders.order_purchase_timestamp >= "2017-01-01") & (orders.order_purchase_timestamp < "2018-09-01")]
rev = reviews.groupby("order_id", as_index=False).review_score.mean()

df = (items.merge(orders, on="order_id")
      .merge(products[["product_id","product_category_name"]], on="product_id", how="left")
      .merge(cats, on="product_category_name", how="left")
      .merge(customers[["customer_id","customer_state"]], on="customer_id", how="left")
      .merge(rev, on="order_id", how="left"))
df["month"] = df.order_purchase_timestamp.dt.to_period("M").astype(str)
df["delivery_days"] = (df.order_delivered_customer_date - df.order_purchase_timestamp).dt.days
df["late"] = df.order_delivered_customer_date > df.order_estimated_delivery_date
df["category"] = df.product_category_name_english.fillna("unknown")
cols = ["order_id","month","customer_state","seller_id","category","price","freight_value","delivery_days","late","review_score"]
df[cols].to_csv("../excel/olist_clean.csv", index=False)
print(len(df))  

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


109880


In [2]:
df[cols].head()
df[cols].isna().sum()

order_id            0
month               0
customer_state      0
seller_id           0
category            0
price               0
freight_value       0
delivery_days       8
late                0
review_score      823
dtype: int64